要运行此程序，请在 **AMD Dev Cloud** 上按“*运行*”并按“*运行全部*”！
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> 如果您需要帮助，请加入 Discord + ⭐ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</a> </i> ⭐
</div>

要在本地设备上安装 Unsloth，请按照 [our guide](https://unsloth.ai/docs/get-started/install) 操作。此笔记本已获得许可 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)。

您将学习如何执行 [data prep](#Data)、如何执行 [train](#Train)、如何执行 [run the model](#Inference) 以及如何保存它

### 消息

隆重推出 **Unsloth Studio** - 一个新的开源、无代码 Web UI，用于训练和运行法学硕士。 [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<表><tr>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~% 2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fupload s%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia% 26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&宽度=376&dpr=3&质量=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>训练模型</b> - 无需代码</sub></td>
<tdalign="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Z q%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26toke n%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&宽度=376&dpr=3&质量=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio 聊天 UI"></a><br><sub><b>在 Mac、Windows 和 Linux 上运行 GGUF 模型</b></sub></td>
</tr></表>

训练 MoE - DeepSeek、GLM、Qwen 和 gpt-oss 速度提高 12 倍，VRAM 减少 35%。 [Blog](https://unsloth.ai/docs/new/faster-moe)

超长上下文强化学习现已推出，上下文窗口增加了 7 倍！ [Blog](https://unsloth.ai/docs/new/grpo-long-context)

强化学习新功能：[FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

请访问我们的文档以获取所有 [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) 和 [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks)。

### 安装

In [ ]:
%%bash
python -m pip install -qU uv --root-user-action=ignore

ROCM_TAG="$({ command -v amd-smi >/dev/null 2>&1 && amd-smi version 2>/dev/null | awk -F'ROCm version: ' 'NF>1{split($2,a,"."); print "rocm"a[1]"."a[2]; ok=1; exit} END{exit !ok}'; } || { [ -r /opt/rocm/.info/version ] && awk -F. '{print "rocm"$1"."$2; exit}' /opt/rocm/.info/version; } || { command -v hipconfig >/dev/null 2>&1 && hipconfig --version 2>/dev/null | awk -F': *' '/HIP version/{split($2,a,"."); print "rocm"a[1]"."a[2]; ok=1; exit} END{exit !ok}'; } || { command -v dpkg-query >/dev/null 2>&1 && ver="$(dpkg-query -W -f='${Version}\n' rocm-core 2>/dev/null)" && [ -n "$ver" ] && awk -F'[.-]' '{print "rocm"$1"."$2; exit}' <<<"$ver"; } || { command -v rpm >/dev/null 2>&1 && ver="$(rpm -q --qf '%{VERSION}\n' rocm-core 2>/dev/null)" && [ -n "$ver" ] && awk -F'[.-]' '{print "rocm"$1"."$2; exit}' <<<"$ver"; })"
[ -n "$ROCM_TAG" ] || { echo "Could not detect ROCm. Install ROCm first or set ROCM_TAG manually."; exit 1; }
case "$ROCM_TAG" in
  rocm6.[0-4]|rocm7.[02]) T="$ROCM_TAG" ;;
  rocm6.*) T="rocm6.4" ;;
  *) T="rocm7.1" ;;
esac
pip install bitsandbytes
PYTORCH_INDEX_URL="https://download.pytorch.org/whl/${T}"
uv pip install --system -U --force-reinstall \
    torch torchvision torchaudio triton-rocm \
    --index-url "$PYTORCH_INDEX_URL"
uv pip install --system cut-cross-entropy torchao --no-deps
uv pip install --system -U --no-deps "unsloth[amd]" "unsloth_zoo[amd]"
uv pip install --system --no-deps -r "$(python -c 'import pathlib,site;print(next(p for r in [*site.getsitepackages(),site.getusersitepackages()] if (p:=pathlib.Path(r,"studio/backend/requirements/no-torch-runtime.txt")).exists()))')" torchao
uv pip install --system --no-deps -U "tokenizers>=0.22.0,<=0.23.0"


In [ ]:
!uv pip install --system -qqq sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer "transformers==4.56.2" jiwer einops addict easydict
!uv pip install --system -qqq --no-deps accelerate peft "trl==0.22.2"


### 不懒惰

我们先准备好我们本地的OCR模型

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("unsloth/DeepSeek-OCR", local_dir = "deepseek_ocr")

In [ ]:
from unsloth import FastVisionModel # 法学硕士的 FastLanguageModel
import torch
from transformers import AutoModel
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = '0'
# 我们支持 4 位预量化模型，下载速度提高 4 倍 + 无 OOM。
fourbit_models = [
    "unsloth/Qwen3-VL-8B-Instruct-bnb-4bit", # Qwen 3 视觉支持
    "unsloth/Qwen3-VL-8B-Thinking-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Instruct-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Thinking-bnb-4bit",
] # 更多模型请访问 https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    "./deepseek_ocr",
    load_in_4bit = False, # 使用4bit来减少内存使用。对于 16 位 LoRA 为假。
    auto_model = AutoModel,
    trust_remote_code = True,
    unsloth_force_compile = True,
    use_gradient_checkpointing = "unsloth", # 长上下文中的真实或“不懒惰”
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.11.1: Fast Deepseekocr patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at ./deepseek_ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# @title 创建评估函数

import json
import os
from typing import Dict
import numpy as np
from jiwer import cer
from tqdm import tqdm
from datasets import load_dataset


def calculate_cer(ref: str, hyp: str) -> float:
    """Helper to calculate CER and convert to percentage."""
    return cer(ref, hyp) * 100


def evaluate_model(
    model,
    tokenizer,
    dataset,
    num_samples: int = 100,
    base_size: int = 1024,
    image_size: int = 640,
    crop_mode: bool = True,
    verbose: bool = True
):
    """
    Runs the model over a subset of the dataset to see how it performs.
    It'll calculate CER stats and save all the predictions.
    """

    results = {
        'cer_scores': [],
        'predictions': [],
        'references': [],
        'sample_indices': []
    }

    # 确保我们不会尝试抽取超出现有数量的样本
    num_samples = min(num_samples, len(dataset))

    # 从数据集中获取均匀分布的样本
    indices = np.linspace(0, len(dataset) - 1, num_samples, dtype = int)

    # 如果冗长，请使用 tqdm 作为进度条
    iterator = tqdm(indices, desc = "Evaluating") if verbose else indices

    for idx in iterator:
        sample = dataset[int(idx)]

        # model.infer 方法需要文件路径，因此我们保存临时图像
        temp_image_path = f"temp_eval_image_{idx}.jpg"
        sample['image_path'].save(temp_image_path)

        prediction = ""
        reference = sample["text"].strip()

        try:
            # 运行实际推理
            prediction = model.infer(
                tokenizer,
                prompt = "<image>\nFree OCR. ",
                image_file = temp_image_path,
                output_path = "temp_output",
                base_size = base_size,
                image_size = image_size,
                crop_mode = crop_mode,
                eval_mode = True,
                save_results = False,
                test_compress = False
            )

            prediction = prediction.strip()

            # 计算核证减排量
            cer_score = calculate_cer(reference, prediction)

            results['cer_scores'].append(cer_score)
            results['predictions'].append(prediction)
            results['references'].append(reference)
            results['sample_indices'].append(int(idx))

        except Exception as e:
            # 不要让一个坏样本破坏整个评估
            print(f"\nError processing sample {idx}: {e}")
            print(f"Reference was: {reference}")
            continue
        finally:
            # 清理临时文件无论成功还是失败
            if os.path.exists(temp_image_path):
                os.remove(temp_image_path)

    # 添加摘要统计数据
    if results['cer_scores']:
        results['mean_cer'] = np.mean(results['cer_scores'])
        results['median_cer'] = np.median(results['cer_scores'])
        results['std_cer'] = np.std(results['cer_scores'])
        results['min_cer'] = np.min(results['cer_scores'])
        results['max_cer'] = np.max(results['cer_scores'])
    else:
        print("警告：没有成功处理任何样本。")
        results['mean_cer'] = -1.0

    results['num_samples'] = len(results['cer_scores'])

    return results

def print_evaluation_summary(results: Dict, title: str = "Evaluation Results"):
    """Prints a nice summary of the stats to the console."""

    print("\n" + "="*60)
    print(f"{title}")
    print("="*60)
    print(f"Number of samples: {results['num_samples']}")
    print(f"Mean CER: {results['mean_cer']:.2f}%")
    print(f"Median CER: {results['median_cer']:.2f}%")
    print(f"Std Dev: {results['std_cer']:.2f}%")
    print(f"Min CER: {results['min_cer']:.2f}%")
    print(f"Max CER: {results['max_cer']:.2f}%")
    print("="*60)

    # 显示最好和最差的例子
    sorted_indices = np.argsort(results['cer_scores'])

    print("\n 最佳预测（最低 CER）：")
    for i in range(min(3, len(sorted_indices))):
        idx = sorted_indices[i]
        print(f"\nSample {results['sample_indices'][idx]} (CER: {results['cer_scores'][idx]:.2f}%)")
        print(f"Reference:  {results['references'][idx][:100]}...")
        print(f"Prediction: {results['predictions'][idx][:100]}...")

    print("\n 最差预测（最高 CER）：")
    for i in range(min(3, len(sorted_indices))):
        idx = sorted_indices[-(i+1)]
        print(f"\nSample {results['sample_indices'][idx]} (CER: {results['cer_scores'][idx]:.2f}%)")
        print(f"Reference:  {results['references'][idx][:100]}...")
        print(f"Prediction: {results['predictions'][idx][:100]}...")

def save_evaluation_results(results: Dict, filepath: str):
    """Save full results dictionary to a JSON file."""
    with open(filepath, 'w', encoding = 'utf-8') as f:
        json.dump(results, f, ensure_ascii = False, indent = 2)
    print(f"\n✅ Results saved to {filepath}")

### 让我们评估一下 Deepseek-OCR 波斯语转录的基线性能

In [ ]:
print("正在加载评估数据集...")
from datasets import load_dataset

eval_dataset = load_dataset("hezarai/parsynth-ocr-200k", split = "test")

print("\n 运行基线评估...")
baseline_results = evaluate_model(
    model = model,
    tokenizer = tokenizer,
    dataset = eval_dataset,
    num_samples = 200,
    base_size = 1024,
    image_size = 640,
    crop_mode = True,
    verbose = True
)

print_evaluation_summary(baseline_results, "Baseline Model Performance")
save_evaluation_results(baseline_results, "baseline_evaluation.json")

Loading evaluation dataset...


README.md:   0%|          | 0.00/967 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/255M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/57.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/179999 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20000 [00:00<?, ? examples/s]


 Running Baseline Evaluation...


Evaluating: 100%|██████████| 200/200 [03:12<00:00,  1.04it/s]


Baseline Model Performance
Number of samples: 200
Mean CER: 149.07%
Median CER: 80.00%
Std Dev: 310.39%
Min CER: 0.00%
Max CER: 3500.00%

 Best Predictions (Lowest CER):

Sample 5024 (CER: 0.00%)
Reference:  چون هستی خیلی زیاد...
Prediction: چون هستی خیلی زیاد...

Sample 3517 (CER: 0.00%)
Reference:  تو ایران هیچوقت از اینها وجود نخواهد داشت...
Prediction: تو ایران هیچوقت از اینها وجود نخواهد داشت...

Sample 9949 (CER: 0.00%)
Reference:  کاش میدونستم هیچی بیخیال...
Prediction: کاش میدونستم هیچی بیخیال...

 Worst Predictions (Highest CER):

Sample 11155 (CER: 3500.00%)
Reference:  خسو...
Prediction: \[ \text{CH}_3\text{CH}_2\text{CH}_2\text{CH}_2\text{CH}_2\text{CH}_2\text{CH}_2\text{CH}_2\text{CH}...

Sample 13366 (CER: 1900.00%)
Reference:  مشو...
Prediction: \[\begin{align*}\underline{\mathfrak{su}}_0\end{align*}\]...

Sample 10552 (CER: 1014.29%)
Reference:  هیییییچ...
Prediction: e
e
e
e
e
e
e
e
e
e
e
e
e
e
e
e
e
e
o
o
o
o
o
o
o
o
o
o
o
o
o
o
o
o
o
o...

✅ Results saved to baselin

<h3>平均基线模型性能：此评估集的字符错误率 (CER) 为 149.07%！</h3>

# 让我们微调 Deepseek OCR！

我们现在添加 LoRA 适配器来进行参数高效微调 - 这使得我们只能有效地训练所有参数的 1%。

**[新]** 我们还支持仅微调模型的视觉部分或仅微调语言部分。或者您可以选择两者！您还可以选择微调注意力或 MLP 层！

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    r = 16,           # 越大，准确率越高，但可能会过拟合
    lora_alpha = 16,  # 建议 alpha == r 至少
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # 我们支持排名稳定的 LoRA
    loftq_config = None, # 和阁楼Q
    # target_modules = "全线性", # 现在可选！如果需要可以指定列表
)

Unsloth: Making `model.base_model.model.model` require gradients


<a名称=“数据”></a>
### 数据准备
我们将使用波斯语 OCR 数据集。目标是将这些图像转换为计算机可读的形式 - 即文本。这对于数字化波斯文本非常有用。

您可以访问数据集[here](https://huggingface.co/datasets/hezarai/parsynth-ocr-200k)。

让我们概览一下数据集。我们将看到第三张图片是什么，以及它的标题是什么。

要格式化数据集，所有视觉微调任务应格式化如下：

```python
[
{ "role": "<|User|>",
  "content": "",
  "images": []
},
{ "role": "<|Assistant|>",
  "content": ""
},
]
```

In [ ]:
from datasets import load_dataset
instruction = "<image>\nFree OCR. "

def convert_to_conversation(sample):
    """Convert dataset sample to conversation format"""
    conversation = [
        {
            "role": "<|User|>",
            "content": instruction,
            "images": [sample['image']]
        },
        {
            "role": "<|Assistant|>",
            "content": sample["text"]
        },
    ]
    return {"messages": conversation}

# 加载数据集
dataset = load_dataset("hezarai/parsynth-ocr-200k", split = "train[:1000]")
dataset = dataset.rename_column("image_path", "image")

让我们将数据集转换为“正确”的格式以进行微调：

In [ ]:
converted_dataset = [convert_to_conversation(sample) for sample in dataset]

我们看看第一个示例的对话是如何构建的：

In [ ]:
converted_dataset[0]

{'messages': [{'role': '<|User|>',
   'content': '<image>\nFree OCR. ',
   'images': [<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=218x48>]},
  {'role': '<|Assistant|>', 'content': 'همهاش جبره و اختیار توهمه'}]}

In [ ]:
# @title 创建数据整理器

import torch
import math
from dataclasses import dataclass
from typing import Dict, List, Any, Tuple
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence
import io

from deepseek_ocr.modeling_deepseekocr import (
    format_messages,
    text_encode,
    BasicImageTransform,
    dynamic_preprocess,
)

@dataclass
class DeepSeekOCRDataCollator:
    """
    Data collator that handles image preprocessing and tokenization at batch time.

    Args:
        tokenizer: Tokenizer instance
        model: Model instance (used to get dtype)
        image_size: Size for image patches (default: 640)
        base_size: Size for global view (default: 1024)
        crop_mode: Whether to use dynamic cropping for large images
        train_on_responses_only: If True, only train on assistant responses (mask user prompts)
    """
    tokenizer: Any
    model: Any
    image_size: int = 640
    base_size: int = 1024
    crop_mode: bool = True
    image_token_id: int = 128815
    train_on_responses_only: bool = True

    def __init__(
        self,
        tokenizer,
        model,
        image_size: int = 640,
        base_size: int = 1024,
        crop_mode: bool = True,
        train_on_responses_only: bool = True,
    ):
        self.tokenizer = tokenizer
        self.model = model
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.image_token_id = 128815
        self.dtype = model.dtype  # 从模型获取 dtype
        self.train_on_responses_only = train_on_responses_only

        self.image_transform = BasicImageTransform(
            mean = (0.5, 0.5, 0.5),
            std = (0.5, 0.5, 0.5),
            normalize = True
        )
        self.patch_size = 16
        self.downsample_ratio = 4

        # 从 tokenizer 获取 BOS 代币 ID
        if hasattr(tokenizer, 'bos_token_id') and tokenizer.bos_token_id is not None:
            self.bos_id = tokenizer.bos_token_id
        else:
            self.bos_id = 0
            print(f"Warning: tokenizer has no bos_token_id, using default: {self.bos_id}")

    def deserialize_image(self, image_data) -> Image.Image:
        """Convert image data (bytes dict or PIL Image) to PIL Image in RGB mode"""
        if isinstance(image_data, Image.Image):
            return image_data.convert("RGB")
        elif isinstance(image_data, dict) and 'bytes' in image_data:
            image_bytes = image_data['bytes']
            image = Image.open(io.BytesIO(image_bytes))
            return image.convert("RGB")
        else:
            raise ValueError(f"Unsupported image format: {type(image_data)}")

    def calculate_image_token_count(self, image: Image.Image, crop_ratio: Tuple[int, int]) -> int:
        """Calculate the number of tokens this image will generate"""
        num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
        num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

        width_crop_num, height_crop_num = crop_ratio

        if self.crop_mode:
            img_tokens = num_queries_base * num_queries_base + 1
            if width_crop_num > 1 or height_crop_num > 1:
                img_tokens += (num_queries * width_crop_num + 1) * (num_queries * height_crop_num)
        else:
            img_tokens = num_queries * num_queries + 1

        return img_tokens

    def process_image(self, image: Image.Image) -> Tuple[List, List, List, List, Tuple[int, int]]:
        """
        Process a single image based on crop_mode and size thresholds

        Returns:
            Tuple of (images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio)
        """
        images_list = []
        images_crop_list = []
        images_spatial_crop = []

        if self.crop_mode:
            # 根据图像大小确定裁剪比例
            if image.size[0] <= 640 and image.size[1] <= 640:
                crop_ratio = (1, 1)
                images_crop_raw = []
            else:
                images_crop_raw, crop_ratio = dynamic_preprocess(
                    image, min_num = 2, max_num = 9,
                    image_size = self.image_size, use_thumbnail = False
                )

            # 使用填充处理全局视图
            global_view = ImageOps.pad(
                image, (self.base_size, self.base_size),
                color = tuple(int(x * 255) for x in self.image_transform.mean)
            )
            images_list.append(self.image_transform(global_view).to(self.dtype))

            width_crop_num, height_crop_num = crop_ratio
            images_spatial_crop.append([width_crop_num, height_crop_num])

            # 处理本地视图（裁剪）（如果适用）
            if width_crop_num > 1 or height_crop_num > 1:
                for crop_img in images_crop_raw:
                    images_crop_list.append(
                        self.image_transform(crop_img).to(self.dtype)
                    )

            # 计算图像标记
            num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
            num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

            tokenized_image = ([self.image_token_id] * num_queries_base + [self.image_token_id]) * num_queries_base
            tokenized_image += [self.image_token_id]

            if width_crop_num > 1 or height_crop_num > 1:
                tokenized_image += ([self.image_token_id] * (num_queries * width_crop_num) + [self.image_token_id]) * (
                    num_queries * height_crop_num)

        else:  # 作物模式=假
            crop_ratio = (1, 1)
            images_spatial_crop.append([1, 1])

            # 对于较小的底座尺寸，请调整尺寸；对于较大的垫
            if self.base_size <= 640:
                resized_image = image.resize((self.base_size, self.base_size), Image.LANCZOS)
                images_list.append(self.image_transform(resized_image).to(self.dtype))
            else:
                global_view = ImageOps.pad(
                    image, (self.base_size, self.base_size),
                    color = tuple(int(x * 255) for x in self.image_transform.mean)
                )
                images_list.append(self.image_transform(global_view).to(self.dtype))

            num_queries = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tokenized_image = ([self.image_token_id] * num_queries + [self.image_token_id]) * num_queries
            tokenized_image += [self.image_token_id]

        return images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio

    def process_single_sample(self, messages: List[Dict]) -> Dict[str, Any]:
            """
            Process a single conversation into model inputs.

            This version builds the token sequence in a single pass,
            accurately calculating the prompt/response split point.
            """

            # --- 1. 设置 ---
            images = []
            for message in messages:
                if "images" in message and message["images"]:
                    for img_data in message["images"]:
                        if img_data is not None:
                            pil_image = self.deserialize_image(img_data)
                            images.append(pil_image)

            if not images:
                raise ValueError("No images found in sample. Please ensure all samples contain images.")

            tokenized_str = []
            images_seq_mask = []
            images_list, images_crop_list, images_spatial_crop = [], [], []

            prompt_token_count = -1 # 开始训练的索引
            assistant_started = False
            image_idx = 0

            # 一开始就添加BOS代币
            tokenized_str.append(self.bos_id)
            images_seq_mask.append(False)

            for message in messages:
                role = message["role"]
                content = message["content"]

                # 检查是否轮到助理
                if role == "<|Assistant|>":
                    if not assistant_started:
                        # 这就是分割点。 *到目前为止*添加的所有代币
                        # 是提示的一部分。
                        prompt_token_count = len(tokenized_str)
                        assistant_started = True

                    # 将 EOS 令牌字符串附加到助手内容的 *end*
                    content = f"{content.strip()} {self.tokenizer.eos_token}"

                # 按图像标记分割此消息的内容
                text_splits = content.split('<image>')

                for i, text_sep in enumerate(text_splits):
                    # 对文本部分进行标记
                    tokenized_sep = text_encode(self.tokenizer, text_sep, bos = False, eos = False)
                    tokenized_str.extend(tokenized_sep)
                    images_seq_mask.extend([False] * len(tokenized_sep))

                    # 如果此文本后跟 <image> 标记
                    if i < len(text_splits) - 1:
                        if image_idx >= len(images):
                            raise ValueError(
                                f"Data mismatch: Found '<image>' token but no corresponding image."
                            )

                        # 处理图像
                        image = images[image_idx]
                        img_list, crop_list, spatial_crop, tok_img, _ = self.process_image(image)

                        images_list.extend(img_list)
                        images_crop_list.extend(crop_list)
                        images_spatial_crop.extend(spatial_crop)

                        # 添加图像占位符标记
                        tokenized_str.extend(tok_img)
                        images_seq_mask.extend([True] * len(tok_img))

                        image_idx += 1 # 移至下一张图片

            # --- 3. 验证和最终准备 ---
            if image_idx != len(images):
                raise ValueError(
                    f"Data mismatch: Found {len(images)} images but only {image_idx} '<image>' tokens were used."
                )

            # 如果我们从未找到助理消息，我们就会处于一种奇怪的状态
            # （例如，仅限用户的提示）。我们掩盖一切。
            if not assistant_started:
                print("警告：示例中未找到辅助消息。屏蔽所有标记。")
                prompt_token_count = len(tokenized_str)

            # 准备图像张量
            images_ori = torch.stack(images_list, dim = 0)
            images_spatial_crop_tensor = torch.tensor(images_spatial_crop, dtype = torch.long)

            if images_crop_list:
                images_crop = torch.stack(images_crop_list, dim = 0)
            else:
                images_crop = torch.zeros((1, 3, self.base_size, self.base_size), dtype = self.dtype)

            return {
                "input_ids": torch.tensor(tokenized_str, dtype = torch.long),
                "images_seq_mask": torch.tensor(images_seq_mask, dtype = torch.bool),
                "images_ori": images_ori,
                "images_crop": images_crop,
                "images_spatial_crop": images_spatial_crop_tensor,
                "prompt_token_count": prompt_token_count, # 现在这是准确的
            }

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        """Collate batch of samples"""
        batch_data = []

        # 处理每个样品
        for feature in features:
            try:
                processed = self.process_single_sample(feature['messages'])
                batch_data.append(processed)
            except Exception as e:
                print(f"Error processing sample: {e}")
                continue

        if not batch_data:
            raise ValueError("No valid samples in batch")

        # 提取列表
        input_ids_list = [item['input_ids'] for item in batch_data]
        images_seq_mask_list = [item['images_seq_mask'] for item in batch_data]
        prompt_token_counts = [item['prompt_token_count'] for item in batch_data]

        # 焊盘序列
        input_ids = pad_sequence(input_ids_list, batch_first = True, padding_value = self.tokenizer.pad_token_id)
        images_seq_mask = pad_sequence(images_seq_mask_list, batch_first = True, padding_value = False)

        # 创建标签
        labels = input_ids.clone()

        # 掩码填充标记
        labels[labels == self.tokenizer.pad_token_id] = -100

        # 遮罩图像标记（模型不应预测这些）
        labels[images_seq_mask] = -100

        # 当 train_on_responses_only = True 时屏蔽用户提示标记（仅训练助理响应）
        if self.train_on_responses_only:
            for idx, prompt_count in enumerate(prompt_token_counts):
                if prompt_count > 0:
                    labels[idx, :prompt_count] = -100

        # 创建注意力蒙版
        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()

        # 准备图像批次（元组列表）
        images_batch = []
        for item in batch_data:
            images_batch.append((item['images_crop'], item['images_ori']))

        # 堆叠空间作物信息
        images_spatial_crop = torch.cat([item['images_spatial_crop'] for item in batch_data], dim = 0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "images": images_batch,
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": images_spatial_crop,
        }

<a name="火车"></a>
### 训练模型
现在让我们训练我们的模型。我们执行 60 个步骤来加快速度，但您可以设置“num_train_epochs=1”以进行完整运行，并关闭“max_steps=None”。我们还支持`DPOTrainer`和`GRPOTrainer`用于强化学习！

我们使用新的“DeepSeekOCRDataCollat​​or”，这将有助于我们的视觉微调设置。

In [ ]:
from transformers import Trainer, TrainingArguments
from unsloth import is_bf16_supported
FastVisionModel.for_training(model) # 启用训练！
data_collator = DeepSeekOCRDataCollator(
    tokenizer = tokenizer,
    model = model,
    image_size = 640,
    base_size = 1024,
    crop_mode = True,
    train_on_responses_only = True,
)
trainer = Trainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = data_collator, # 必须用！
    train_dataset = converted_dataset,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        # num_train_epochs = 1, # 设置此值而不是 max_steps 以进行完整的训练运行
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        fp16 = not is_bf16_supported(),  # 如果不支持 bf16，请使用 fp16
        bf16 = is_bf16_supported(),  # 如果支持，请使用 bf16
        output_dir = "outputs",
        report_to = "none",     # 对于权重和偏差
        dataloader_num_workers = 2,
        # 您必须放置以下项目进行视力微调：
        remove_unused_columns = False,
    ),
)

/tmp/ipython-input-4060472171.py:12: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# @title 显示当前内存统计信息
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA L4. Max memory = 22.161 GB.
8.012 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 77,509,632 of 3,413,615,872 (2.27% trained)
Unsloth: Not an error, but DeepseekOCRForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss
1,4.054700
2,2.636700
3,3.507600
4,3.796600
5,2.209600
6,4.074400
7,2.203600
8,1.928200
9,3.159700
10,3.102700


Unsloth: Will smartly offload gradients to save VRAM!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


### 现在微调后让我们评估模型！

In [ ]:
FastVisionModel.for_inference(model) # 启用推理！

finetuned_results = evaluate_model(
    model = model,
    tokenizer = tokenizer,
    dataset = eval_dataset,
    num_samples = 200,
    base_size = 1024,
    image_size = 640,
    crop_mode = True,
    verbose = True
)

print_evaluation_summary(finetuned_results, "Fine-tuned Model Performance")
save_evaluation_results(finetuned_results, "finetuned_evaluation.json")

Evaluating: 100%|██████████| 200/200 [06:11<00:00,  1.86s/it]


Fine-tuned Model Performance
Number of samples: 200
Mean CER: 60.43%
Median CER: 50.00%
Std Dev: 80.63%
Min CER: 0.00%
Max CER: 916.67%

 Best Predictions (Lowest CER):

Sample 301 (CER: 0.00%)
Reference:  باشه بابا تو لاکچری، تو خاص، تو خفن...
Prediction: باشه بابا تو لاکچری، تو خاص، تو خفن...

Sample 2512 (CER: 0.00%)
Reference:  از شخص حاج عبدالله زنجبیلی میگیرنش...
Prediction: از شخص حاج عبدالله زنجبیلی میگیرنش...

Sample 2713 (CER: 0.00%)
Reference:  نمی دونم والا تحمل نقد ندارن ظاهرا...
Prediction: نمی دونم والا تحمل نقد ندارن ظاهرا...

 Worst Predictions (Highest CER):

Sample 14270 (CER: 916.67%)
Reference:  ۴۳۵۹۴۷۴۷۳۸۹۰...
Prediction: پروپریپریپریپریپریپریپریپریپریپریپریپریپریپریپریپریپریپریپیپریپریپریپریپریپریپریپریپریپریپریپریپریپر...

Sample 3919 (CER: 380.00%)
Reference:  ۷۵۵۰۷۱۰۶۵۹...
Prediction: وادووووووووووووووووووووووووووووووووووو...

Sample 3718 (CER: 333.33%)
Reference:  ۳۲۶۷۲۲۶۵۵۸۴۶...
Prediction: پُپُسوپُسوپُسوپُسوپُسوپُسوپُسوپُسوپُسوپُ...

✅ Results saved to fin

### 现在让我们比较一下两者。

In [ ]:
print("\n" + "="*60)
print("📈 性能比较")
print("="*60)
print(f"Baseline Mean CER:    {baseline_results['mean_cer']:.2f}%")
print(f"Fine-tuned Mean CER:  {finetuned_results['mean_cer']:.2f}%")

improvement = baseline_results['mean_cer'] - finetuned_results['mean_cer']
relative_improvement = (improvement / baseline_results['mean_cer']) * 100

print(f"\n✨ Absolute Improvement: {improvement:.2f}%")
print(f"✨ Relative Improvement: {relative_improvement:.2f}%")
print("="*60)


📈 PERFORMANCE COMPARISON
Baseline Mean CER:    149.07%
Fine-tuned Mean CER:  60.43%

✨ Absolute Improvement: 88.64%
✨ Relative Improvement: 59.46%


仅用 60 个步骤，我们就将字符错误率 (CER) 从 149.07% 降低到 60.43%，字符错误绝对改善了 88.6%！

In [ ]:
# @title 显示最终内存和时间统计数据
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="推理"></a>
### 推论
让我们运行模型吧！

In [ ]:
eval_dataset[0]['image_path'].save('your_image.jpg')
prompt = "<image>\nFree OCR. "
image_file = 'your_image.jpg'
output_path = 'your/output/dir'

# 小：base_size = 512，image_size = 512，crop_mode = False
# 小：base_size = 640，image_size = 640，crop_mode = False
# 基础：base_size = 1024，image_size = 1024，crop_mode = False
# 大：base_size = 1280，image_size = 1280，crop_mode = False

# 高达：base_size = 1024，image_size = 640，crop_mode = True

res = model.infer(tokenizer, prompt = prompt, image_file = image_file,
    output_path = output_path,
    image_size = 640,
    base_size = 1024,
    crop_mode = True,
    save_results = True,
    test_compress = False)

پاکستان
===============save results:===============


image: 0it [00:00, ?it/s]
other: 0it [00:00, ?it/s]


<a name="保存"></a>
### 保存、加载微调模型
要将最终模型保存为 LoRA 适配器，请使用 Hugging Face 的 `push_to_hub` 进行在线保存，或使用 `save_pretrained` 进行本地保存。

**[注意]** 这仅保存 LoRA 适配器，而不是完整模型。要保存到 16 位或 GGUF，请向下滚动！

In [ ]:
model.save_pretrained("deepseek_ocr_lora")  # 本地储蓄
tokenizer.save_pretrained("deepseek_ocr_lora")
# model.push_to_hub("your_name/deepseek_ocr_lora", token = "YOUR_HF_TOKEN") # 在线保存
# tokenizer.push_to_hub("your_name/deepseek_ocr_lora", token = "YOUR_HF_TOKEN") # 在线保存

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

现在，如果您想加载我们刚刚保存用于推理的 LoRA 适配器，请将“False”设置为“True”：

In [ ]:
if False:
    from unsloth import FastVisionModel
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = "deepseek_ocr_lora", # 您用于训练的模型
        load_in_4bit = False, # 使用4bit来减少内存使用。对于 16 位 LoRA 为假。
        auto_model = AutoModel,
        trust_remote_code = True,
        unsloth_force_compile = True,
        use_gradient_checkpointing = "unsloth", # 长上下文中的真实或“不懒惰”
    )
    FastVisionModel.for_inference(model) # 启用推理！

prompt = "<image>\nFree OCR. "
image_file = 'your_image.jpg'
output_path = 'your/output/dir'

# 小：base_size = 512，image_size = 512，crop_mode = False
# 小：base_size = 640，image_size = 640，crop_mode = False
# 基础：base_size = 1024，image_size = 1024，crop_mode = False
# 大：base_size = 1280，image_size = 1280，crop_mode = False

# 高达：base_size = 1024，image_size = 640，crop_mode = True

res = model.infer(tokenizer, prompt = prompt, image_file = image_file,
    output_path = output_path,
    image_size = 640,
    base_size = 1024,
    crop_mode = True,
    save_results = True,
    test_compress = False)

### 保存为 VLLM 的 float16

我们还支持直接保存到`float16`。为 float16 选择“merged_16bit”。使用 `push_to_hub_merged` 上传到您的 Hugging Face 帐户！您可以前往 https://huggingface.co/settings/tokens 获取您的个人代币。有关更多部署选项，请参阅 [our docs](https://unsloth.ai/docs/basics/inference-and-deployment)。

In [ ]:
# 仅选择 1 个进行保存！ （两者都不需要！）

# 本地保存为16位
if False: model.save_pretrained_merged("unsloth_finetune", tokenizer,)

# 导出并保存到您的 Hugging Face 帐户
if False: model.push_to_hub_merged("YOUR_USERNAME/unsloth_finetune", tokenizer, token = "YOUR_HF_TOKEN")

我们就完成了！如果您对 Unsloth 有任何疑问，我们有 [Discord](https://discord.gg/unsloth) 频道！如果您发现任何错误或想要随时更新最新的 LLM 内容，或需要帮助、加入项目等，请随时加入我们的 Discord！

其他一些资源：
1.训练自己的推理模型——Llama GRPO笔记本[Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. 将微调保存到Ollama。 [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 视觉微调 - 射线照相用例。 [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. 请参阅我们的 [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks) 上的 DPO、ORPO、持续预训练、对话微调等笔记本！

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  如果您需要帮助，请加入 Discord + ⭐️ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</i> ⭐️

  此笔记本和所有 Unsloth 笔记本均已获得许可 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)
</div>